<a href="https://colab.research.google.com/github/john891212-oss/AIFFEL_quest_eng/blob/main/NLP/NLP01/Day1_RAG_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [61]:
import os


In [10]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [11]:
#! curl ipinfo.io

In [12]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [13]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [14]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

/tmp/ipykernel_10604/128205925.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [15]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [16]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [17]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [18]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [19]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [20]:
from langchain_community.document_loaders import WebBaseLoader

In [60]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [22]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [23]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [24]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [25]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [26]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [27]:
!pip install tiktoken

In [28]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [29]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [30]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [31]:
client = openai.OpenAI()

In [32]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [33]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [34]:
#!curl ipinfo.io

In [35]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [36]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [37]:
print(embeddings[1])

[0.006435394287109375, -0.0208587646484375, -0.0216064453125, 0.0235595703125, -0.0428466796875, -0.004608154296875, 0.03973388671875, 0.05267333984375, -0.0018892288208007812, -0.0237579345703125, 0.0137939453125, 0.0159912109375, -0.044464111328125, -0.00013935565948486328, -0.007648468017578125, 0.02557373046875, 0.0254058837890625, 0.035491943359375, -0.0517578125, 0.01232147216796875, 0.0248260498046875, -0.00045800209045410156, 0.0193939208984375, 0.07525634765625, -0.0020503997802734375, -0.00921630859375, 0.016815185546875, -0.007152557373046875, 0.02655029296875, -0.048492431640625, 0.043182373046875, -0.04254150390625, 0.01165008544921875, -0.032867431640625, -0.0333251953125, -0.046142578125, 0.0019063949584960938, 0.0225677490234375, -0.01812744140625, 0.03253173828125, 0.0293121337890625, 0.011749267578125, -0.01442718505859375, 0.0030574798583984375, 0.0243377685546875, 0.048187255859375, -0.01355743408203125, 0.049346923828125, -0.040618896484375, 0.04400634765625, 0.045

In [38]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [39]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [40]:
query = ["this is red fruit"]

In [41]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7477942151308524
0.48995458116791124
0.4084112226829356


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [42]:
!pip install chromadb

In [43]:
!pip install langchain-chroma

In [44]:
from langchain_chroma import Chroma

In [45]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [46]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [47]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [48]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [49]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [50]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [51]:
!pip install -U langchain langchain-classic

In [52]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [53]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [54]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [55]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_10604/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His appearance is unique and different from others, almost like an animal, a spirit, or an image. There is a suggestion of a feminine element in his features, and his face seems to express a deep, quiet, almost fanatical yet passionate absorption. Overall, Demian's appearance is described as being unimaginably different from the rest of the people around him.

마크다운 형식으로 출력해봅니다

In [56]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His appearance is unique and different from others, almost like an animal, a spirit, or an image. There is a suggestion of a feminine element in his features, and his face seems to express a deep, quiet, almost fanatical yet passionate absorption. Overall, Demian's appearance is described as being unimaginably different from the rest of the people around him.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [57]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


"Demian: The Story of Emil Sinclair's Youth" is a novel by Hermann Hesse, first published in 1919. The book is a coming-of-age story that explores themes of self-discovery, spirituality, and the duality of human nature. Since "Demian" is a literary character and not a visual image or person with a detailed depiction, how he "looks" is largely left to the reader's imagination.

In the novel, Demian is characterized by his mysterious and profound demeanor. He is described as having a mature and enigmatic presence, which distinguishes him from his peers. While no specific physical details are extensively described in the book, his personality and the influence he has on the protagonist, Emil Sinclair, are central to the narrative.

Readers might imagine Demian based on his intellectual and almost mystical influence, often perceiving him with a sense of allure and intrigue rather than focusing on physical attributes.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


# 필요한 라이브러리를 모두 다운받습니다  

In [58]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [59]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters

In [ ]:
!pip install chromadb

In [ ]:
!pip install langchain-chroma

In [ ]:
!pip install -U langchain langchain-classic

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [62]:
import os

In [63]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [74]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)
### 창의적인 대답을 원하면 1에 가깝게 설정
## 요약이나 RAG용은 0으로 설정

In [ ]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

Text splitter 사용을 위한 준비입니다

In [65]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Step 1 Document loader

pdf 파일 페이지수 만큼 document가 생성된다

In [69]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()
documents = loader.load()


182개 문서와 다운로드소스를 알 수 있다.

In [72]:
print("문서 개수:", len(documents))
print("첫 페이지 일부:")
print(documents[2].page_content[:500])
print("메타데이터:")
print(documents[0].metadata)

문서 개수: 182
첫 페이지 일부:
Prologue 
I cannot tell my story without going a long way back. 
If it were possible I would go back much farther still to 
the very earliest years of my childhood and beyond them 
to my family origins. 
When poets write novels they are apt to behave as if 
they were gods, with the power to look beyond and com­
prehend any human story and serve it up as if the 
Almighty himself, omnipresent, were relating it in all 
its naked truth. That I am no more able to do than the 
poets. But my story is m
메타데이터:
{'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}


### Step 2 Text splitters

In [73]:
splits = text_splitter.split_documents(documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

임베딩 모델를 로드 합니다.

In [75]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

벡터화 내지 수치화를 해서 VectorStore에 저장합니다.  
쿼리와 유사한 답변을 찾을수 있을껍니다.

In [76]:
db = Chroma.from_documents(docs, embedding_model)

### Step 4 Retrievers

In [94]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

In [93]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

### Step 5 Question Answering

In [96]:
query = "What is Sinclair's personality like in Demian?"
result = qa(query)

In [97]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

In "Demian," Sinclair's personality is complex and evolves throughout the narrative. Initially, he is portrayed as introspective and somewhat insecure, grappling with his identity and the expectations of society. He experiences a deep inner conflict between the conventional world and his desire for self-discovery and authenticity. As the story progresses, Sinclair becomes more open-minded and receptive to new ideas, particularly those presented by Demian and Frau Eva. He shows a growing sense of individuality and a quest for deeper meaning in life, reflecting a desire to break free from societal norms. Sinclair's character embodies themes of transformation, self-exploration, and the search for personal truth.

In [98]:
query = "데미안이라는 작품내에서 싱클레어의 성격은 어떠한가?"
result = qa(query)

In [99]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

"데미안"에서 싱클레어의 성격은 복잡하고 내적인 갈등을 겪는 인물로 묘사됩니다. 그는 어린 시절의 순수함과 성숙함 사이에서 갈등하며, 자신의 정체성과 욕망을 탐구하는 과정에서 고뇌합니다. 싱클레어는 외부 세계와의 단절을 느끼고, 꿈과 환상에 빠져들며, 사랑과 성에 대한 갈망을 가지고 있지만 이를 충족시키지 못해 괴로워합니다. 또한, 그는 데미안과의 관계를 통해 자신을 발견하고 성장해 나가는 모습을 보입니다. 전반적으로 싱클레어는 내면의 갈등과 자아 탐색을 통해 성숙해가는 인물로 그려집니다.

## step6 룰북을 이해하는 챗봇



NBA룰북 다온로드 및 pdf파일 로드

In [77]:
import os
import requests

os.makedirs("data", exist_ok=True)

pdf_url = "https://cdn.nba.com/manage/2026/01/Official-2025-26-NBA-Playing-Rules.pdf"
pdf_path = "data/nba_2025_26_rulebook.pdf"

response = requests.get(pdf_url)
response.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("다운로드 완료:", pdf_path)
print("파일 크기:", os.path.getsize(pdf_path), "bytes")

다운로드 완료: data/nba_2025_26_rulebook.pdf
파일 크기: 1653286 bytes


In [78]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("로드된 페이지 수:", len(documents))
print("첫 페이지 일부:")
print(documents[0].page_content[:500])
print("metadata:")
print(documents[0].metadata)

로드된 페이지 수: 76
첫 페이지 일부:
This Page Intentionally Left Blank  
It is here to hold a place for cover for screen version.  
DO NOT INCLUDE AS PART OF PRINT FILE!
2025-26 
OFFICIALRULES
metadata:
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'creationdate': '2025-09-25T13:08:06-04:00', 'moddate': '2025-09-25T13:08:07-04:00', 'trapped': '/False', 'source': 'data/nba_2025_26_rulebook.pdf', 'total_pages': 76, 'page': 0, 'page_label': '1'}


splitter로 chunking

In [100]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", "Section", "Rule", ". ", " ", ""]
)

splits = text_splitter.split_documents(documents)

print("chunk 개수:", len(splits))
print(splits[15].page_content[:500])
print(splits[0].metadata)

chunk 개수: 280
administrator will be stationed at the scorer’s table to facilitate communication between the Replay 
Center Official, on-court game officials, official scorer, and other personnel at the scorer’s table. 
All officials and the courtside administrator shall be approved by the League Office.
b. The officials shall wear the uniform prescribed by the NBA.
Section II—Duties of the Officials
a. The officials shall, prior to the start of the game, inspect and approve all equipment,  
including court, b
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'creationdate': '2025-09-25T13:08:06-04:00', 'moddate': '2025-09-25T13:08:07-04:00', 'trapped': '/False', 'source': 'data/nba_2025_26_rulebook.pdf', 'total_pages': 76, 'page': 0, 'page_label': '1'}


vector store 생성

In [80]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_nba_rules"
)

print("Vector Store 생성 완료")

Vector Store 생성 완료


In [81]:
vectorstore = Chroma(
    persist_directory="./chroma_nba_rules",
    embedding_function=embeddings
)

Retrievers

In [82]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.6,
    }
)

In [101]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=700,
)

langchain

In [91]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 NBA 룰 설명 도우미입니다.
아래 context는 NBA 공식 룰북에서 검색된 내용입니다.

답변 규칙:
1. 반드시 context에 근거해서 답하세요.
2. 한국어로 쉽게 설명하세요.
3. 가능하면 "룰북상 핵심", "예시", "주의할 점"으로 나누세요.
5. 임의로 FIBA, KBL, NCAA 룰을 섞지 마세요.
6. 마지막에 참고한 page metadata를 간단히 표시하세요.

Question:
{question}

Context:
{context}
""")

Q % A

In [92]:
question = "NBA에서 개인 파울 몇 개면 퇴장인가요?"
answer = rag_chain.invoke(question)

print(answer)

제공된 룰북 근거만으로는 확정하기 어렵습니다. 개인 파울의 수에 따른 퇴장 규정에 대한 정보는 포함되어 있지 않습니다. 

참고한 page metadata: [문서 1 | page=69 | source=data/nba_2025_26_rulebook.pdf], [문서 2 | page=43 | source=data/nba_2025_26_rulebook.pdf], [문서 3 | page=14 | source=data/nba_2025_26_rulebook.pdf], [문서 4 | page=49 | source=data/nba_2025_26_rulebook.pdf]


영어 룰북에 대해 한국어 질문을 이해하지 못한거 같아서 한/영 퀘스천을 두개로 입력해 보았다.

In [102]:
question = "NBA에서 개인 파울 몇 개면 퇴장인가요? personal foul sixth personal foul disqualified disqualification"
docs = retriever.invoke(question)

for i, doc in enumerate(docs):
    print(f"\n--- 검색 결과 {i+1} ---")
    print(doc.page_content[:1000])
    print(doc.metadata)


--- 검색 결과 1 ---
who was disqualified by reason of receiving six personal fouls. Each subsequent requirement 
to replace an injured or ejected player will be treated in this inverse order. Any such re-entry 
into a game by a disqualified player shall be penalized by a technical foul.
c. In the event that a player leaves the playing court while the ball is in play , play will  
continue until the next stoppage of play and the player will be replaced if he is not ready to  
return. No technical foul will be assessed, but the incident will be reviewed by the League  
Office for a possible fine and/or suspension.
EXCEPTION: Rule 10, Section XV
Section II—Starting Line-Ups
At least 30 minutes before the game is scheduled to begin, the scorers shall be supplied 
with the name and number of each player who will start the game. Failure to comply with this 
provision shall be reported to the League Office.
Section III—The Captain
a. A team may have a captain and a co-captain numbering a maximum

A flagrant foul—penalty (1) is unnecessary contact committed by a player against an  
opponent.
A flagrant foul—penalty (2) is unnecessary and excessive contact committed by a  
player against an opponent. It is an unsportsmanlike act and the offender is ejected following  
confirmation by instant replay review .   
같은 파울 규정을 찾았고 이에 대한 대답을 영어로 한번.   
한국어로도 답하게 하였다.

In [103]:
search_query = "personal foul sixth personal foul disqualified NBA rulebook"

docs = retriever.invoke(search_query)

for i, doc in enumerate(docs):
    print(f"\n--- 검색 결과 {i+1} ---")
    print(doc.page_content[:1200])
    print(doc.metadata)


--- 검색 결과 1 ---
try to dribble by an opponent unless there is a reasonable chance of getting by without contact.
B. FOULS: FLAGRANT—UNSPORTSMANLIKE
T o be unsportsmanlike is to act in a manner unbecoming to the image of professional  
basketball. It consists of acts of deceit, disrespect of officials and profanity . The penalty for 
such action is a technical foul. Repeated acts shall result in expulsion from the game and a  
minimum fine of $2,000.
A flagrant foul—penalty (1) is unnecessary contact committed by a player against an  
opponent.
A flagrant foul—penalty (2) is unnecessary and excessive contact committed by a  
player against an opponent. It is an unsportsmanlike act and the offender is ejected following  
confirmation by instant replay review .
The offender will be subject to a fine not exceeding $100,000 and/or suspension by 
the Commissioner.
See Rule 12B, Section IV for interpretation and penalties.
C. BLOCK-CHARGE
{'trapped': '/False', 'total_pages': 76, 'creationdat

In [104]:
question_ko = "NBA에서 개인 파울 몇 개면 퇴장인가요?"

context = format_docs(docs)

answer = final_chain.invoke({
    "question": question_ko,
    "context": context
})

print(answer)

NBA에서 개인 파울로 퇴장당하는 기준은 6개의 개인 파울을 받는 것입니다. 즉, 한 선수가 6번째 개인 파울을 범하면 그 선수는 경기에서 퇴장하게 됩니다.


룰북 같은 경우 영어 핵심 키워드가 질문에 들어가 있어야 context를 근거로 한 답변을 얻을수 있었다.

In [105]:
question_ko = "NBA 벌금 기준?"

context = format_docs(docs)

answer = final_chain.invoke({
    "question": question_ko,
    "context": context
})

print(answer)

NBA에서 벌금 기준은 다음과 같습니다:

1. **비신사적인 행동**: 비신사적인 행동을 할 경우, 기술적 파울이 부여되며, 반복적인 경우에는 경기에서 퇴장당하고 최소 $2,000의 벌금이 부과됩니다.

2. **플래그런트 파울**:
   - **1종**: 불필요한 접촉이 발생한 경우, 해당 선수는 퇴장당하지 않지만, 벌금이 부과될 수 있습니다.
   - **2종**: 불필요하고 과도한 접촉이 발생한 경우, 해당 선수는 퇴장당하며, 최대 $100,000의 벌금과 함께 커미셔너에 의해 출장 정지 처분을 받을 수 있습니다.

3. **특정 경기에서의 퇴장**: 시즌 중 토너먼트 챔피언십 경기나 플레이인 경기에서 퇴장당할 경우, $2,000의 벌금이 부과됩니다.

4. **교전 중 벤치에서 이탈**: 교전 중 벤치에서 이탈한 선수는 최대 $100,000의 벌금과 함께 1경기 출장 정지 처분을 받을 수 있습니다.

이와 같은 규정들은 NBA의 이미지와 스포츠맨십을 유지하기 위한 것입니다.


In [109]:
question_ko = "NBA three-point line distance?"

context = format_docs(docs)

answer = final_chain.invoke({
    "question": question_ko,
    "context": context
})

print(answer)

제공된 context에는 NBA의 3점 라인 거리와 관련된 정보가 포함되어 있지 않습니다. 따라서 이 질문에 대한 답변을 드릴 수 없습니다. 추가적인 정보가 필요하시면 말씀해 주세요!


In [108]:
question_ko = "NBA 총 경기 수?"

context = format_docs(docs)

answer = final_chain.invoke({
    "question": question_ko,
    "context": context
})

print(answer)

제공된 context에는 NBA의 총 경기 수에 대한 정보가 포함되어 있지 않습니다. 따라서 이 질문에 대한 답변을 드릴 수 없습니다. NBA의 총 경기 수에 대한 정보는 다른 출처에서 확인해야 할 것 같습니다.


갑자기 아몰랑 못이 되었다.

In [110]:
query = "three-point field goal line 23 feet 9 inches 22 feet NBA rulebook"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- 검색 결과 {i+1} ---")
    print(doc.page_content[:1200])
    print(doc.metadata)


--- 검색 결과 1 ---
19 FEET TO FREE THROW LINE (OUTSIDE)
18 FEET 10 INCHES (INSIDE)
13 FEET (inside)
6 INCHES
6 INCHES
18 IN. RADIUS
(INSIDE)
6 FEET RADIUS
(OUTSIDE)
15.5”
12.29”
3 FEET (outside)
14 FEET
28 FEET (INSIDE)
WIDTH 50 FEET (inside)
EACH QUADRANT 19 FEET (OUTSIDE)
OF QUADRANTS
3 FEET
4 FEET20 FEET 11 INCHES
15 FEET
{'producer': 'Adobe PDF Library 17.0', 'source': 'data/nba_2025_26_rulebook.pdf', 'page': 7, 'moddate': '2025-09-25T13:08:07-04:00', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'creationdate': '2025-09-25T13:08:06-04:00', 'trapped': '/False', 'page_label': '8', 'total_pages': 76}

--- 검색 결과 2 ---
- 36 -
b. Allowance may be made for a player who, having been in this area for less than 
three seconds, is in the act of shooting at the end of the third second. Under these conditions, 
the 3-second count is discontinued while his continuous motion is toward the basket. If that 
continuous motion ceases, the previous 3-second count is continued. This is also true if it 


In [112]:
question_ko = "NBA에서 3점 슛 라인은 림에서 얼마나 떨어져 있나요?"
print(answer)

제공된 context에는 NBA의 3점 라인 거리와 관련된 정보가 포함되어 있지 않습니다. 따라서 이 질문에 대한 답변을 드릴 수 없습니다. 추가적인 정보가 필요하시면 말씀해 주세요!


질문의 의도를 여러방향에서 파악해서 답변하도록 해보자

In [113]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

multi_query_prompt = ChatPromptTemplate.from_template("""
너는 NBA 룰북 검색을 돕는 어시스턴트다.

사용자의 질문을 보고, NBA 공식 영어 룰북에서 찾기 좋은 검색어를 3~5개 만들어라.

규칙:
- 영어 검색어만 작성
- 한 줄에 하나씩 작성
- 사용자가 짧게 말해도 가능한 의도를 넓게 추론
- 너무 긴 문장은 피하고 핵심 룰 용어를 포함

사용자 질문:
{question}
""")

multi_query_chain = multi_query_prompt | llm | StrOutputParser()

여러 검색어 사용

In [114]:
def generate_search_queries(question):
    result = multi_query_chain.invoke({"question": question})
    queries = [q.strip("- ").strip() for q in result.split("\n") if q.strip()]
    return queries

In [115]:
def retrieve_multi_query(question, k_per_query=3):
    queries = generate_search_queries(question)

    all_docs = []
    seen = set()

    for query in queries:
        docs = vectorstore.similarity_search(query, k=k_per_query)

        for doc in docs:
            key = (doc.metadata.get("source"), doc.metadata.get("page"), doc.page_content[:100])
            if key not in seen:
                seen.add(key)
                all_docs.append(doc)

    return queries, all_docs

질문 중간 대답 정리

In [117]:
intent_prompt = ChatPromptTemplate.from_template("""
사용자의 NBA 룰 질문을 분석하라.

다음 형식으로만 답하라.

의도:
관련 룰 키워드:
주의할 점:

사용자 질문:
{question}
""")

intent_chain = intent_prompt | llm | StrOutputParser()

In [118]:
print(intent_chain.invoke({"question": "3점 슛 라인"}))

의도: 3점 슛 라인의 위치와 규칙에 대한 이해
관련 룰 키워드: 3점 슛, 슛 라인, 거리
주의할 점: 3점 슛 라인은 코트의 중앙에서부터 일정 거리(23.75피트 또는 22피트) 떨어져 있으며, NBA와 FIBA의 규칙이 다를 수 있으므로 구분해야 함.


깔끔한 챗봇이란 무엇인가?   
사람스러움 첨가...

In [119]:
answer_prompt = ChatPromptTemplate.from_template("""
너는 NBA 룰을 쉽게 풀어주는 농구 해설자다.

사용자 질문:
{question}

질문 의도 분석:
{intent}

검색에 사용한 영어 쿼리:
{queries}

NBA 공식 룰북에서 찾은 근거:
{context}

답변 방식:
- 먼저 결론부터 짧게 말한다.
- 그 다음 농구 상황으로 쉽게 설명한다.
- 룰북 근거가 있는 내용과 일반 해설을 구분한다.
- 사용자가 헷갈릴 만한 포인트를 먼저 짚어준다.
- 너무 딱딱한 법조문 말투는 피한다.
- 단, 룰북에 없는 내용을 확정적으로 말하지 않는다.
- 한국어로 답한다.

답변:
""")

In [120]:
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs):
        page = doc.metadata.get("page", "unknown")
        source = doc.metadata.get("source", "unknown")
        formatted.append(
            f"[문서 {i+1} | page={page} | source={source}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [121]:
def nba_rule_chatbot_flexible(question):
    # 1. 질문 의도 분석
    intent = intent_chain.invoke({"question": question})

    # 2. 여러 검색어 생성 + 검색
    queries, docs = retrieve_multi_query(question, k_per_query=3)

    # 3. context 구성
    context = format_docs(docs)

    # 4. 답변 생성
    chain = answer_prompt | llm | StrOutputParser()

    answer = chain.invoke({
        "question": question,
        "intent": intent,
        "queries": "\n".join(queries),
        "context": context,
    })

    return answer

In [122]:
print(nba_rule_chatbot_flexible("수비수가 페인트존에 오래 있으면 반칙이야?"))

결론부터 말하자면, 수비수가 페인트존(3초 구역)에 3초 이상 머물면 반칙입니다. 

이제 농구 상황으로 쉽게 설명해볼게요. 만약 수비수가 페인트존에 서서 공격수를 지키지 않고 가만히 있다면, 3초가 지나면 반칙이 됩니다. 하지만 만약 공격수가 페인트존에 들어와서 수비수가 그를 방어하고 있다면, 수비수는 그 공격수를 적극적으로 방어해야 해요. 만약 공격수가 페인트존을 떠나면, 수비수는 다시 3초를 계산할 수 있습니다.

여기서 헷갈릴 수 있는 점은, 수비수가 공격수를 방어하고 있을 때는 3초 카운트가 멈춘다는 거예요. 즉, 수비수가 공격수를 잘 방어하고 있다면, 3초를 넘겨도 반칙이 아닙니다. 하지만 방어를 하지 않으면 바로 반칙이 되죠.

NBA 룰북에 따르면, 수비수가 페인트존에 있을 때는 공격수가 공을 가지고 있는 상황에서만 3초 카운트가 시작된다고 해요. 만약 공격수가 공을 패스하거나 페인트존을 떠나면, 수비수는 다시 3초를 계산할 수 있습니다. (출처: [문서 1 | page=35], [문서 8 | page=35])

이렇게 이해하면 수비수의 위치와 행동에 따라 반칙 여부가 결정된다는 점을 기억해 주세요!


맨 처음 프롬프트에선 context에만 근거하라고 해서 아몰랑 봇이 되었고    
자유도와 역활을 새로 준 프롬프트에선 좀 더 자세한 설명을 룰북을 근거로 하고 있다.

chunk size나 질문의 수, 의도파악, retriever 셋팅들을 조절하며 챗봇을 만들어 보았습니다.